# GarsonBot Colab Qwen3 0.6B LoRA Smoke Test

## A. Title And Warning
This notebook is the first executable Google Colab notebook for the `robot_waiter_ai` fine-tuning smoke test.

Important warnings:
- This is a smoke-test notebook, not production training.
- This notebook downloads a model only when you run it inside Google Colab.
- No model has been trained yet in this repository.
- No model should be downloaded or trained locally from this repo task.
- `Qwen/Qwen3-0.6B` is only the first workflow-validation target, not the final production model.

Project scope reminders:
- Turkish conversational AI module for a waiter robot
- no ROS, navigation, SLAM, motor control, FastAPI, speech runtime, database, or dashboard work here
- Jetson Orin NX remains a future inference target, not the Colab training target


## B. Runtime Check
Use a Google Colab GPU runtime. GPU availability varies by session and Colab plan.

If `nvidia-smi` shows no GPU, stop and switch the runtime before continuing.


In [ ]:
import platform
import subprocess
import sys

print('Python version:', sys.version)
print('Platform:', platform.platform())

try:
    result = subprocess.run(['nvidia-smi'], check=False, capture_output=True, text=True)
    print(result.stdout if result.stdout else result.stderr)
    if result.returncode != 0:
        print('WARNING: No working GPU detected. Switch Colab to a GPU runtime before training.')
except FileNotFoundError:
    print('WARNING: nvidia-smi is not available. This usually means no GPU runtime is attached.')


## C. Project Setup
Choose one project setup option below.

Use either:
- Option 1: upload a project zip manually
- Option 2: mount Google Drive

Do not run both setup paths unless you intentionally want to overwrite the working copy.


### Option 1: Upload Project Zip Manually
Create a zip of the current repository on your local machine, upload it to Colab, then extract it.


In [ ]:
# Option 1 only
from google.colab import files

uploaded = files.upload()
print('Uploaded files:', list(uploaded.keys()))


In [ ]:
# Option 1 only
# Update the zip name here if needed.
!mkdir -p /content/Garson-bot
!unzip -q Garson-bot.zip -d /content/Garson-bot
%cd /content/Garson-bot
!pwd


### Option 2: Mount Google Drive
Put the repo folder or a repo zip in Google Drive, mount it in Colab, and work from there.


In [ ]:
# Option 2 only
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
# Option 2 only
# Update this path if your repo folder name or Drive location is different.
%cd /content/drive/MyDrive/Garson-bot
!pwd

# Optional: copy the repo to /content for faster local Colab disk access.
# !cp -r /content/drive/MyDrive/Garson-bot /content/Garson-bot
# %cd /content/Garson-bot


## D. Install Dependencies
This install cell is for Colab only. Do not add these packages to local `requirements.txt` for this repo task.


In [ ]:
!pip install -q -U transformers datasets peft trl accelerate bitsandbytes sentencepiece


## E. Dataset Path Configuration
This section verifies the existing processed dataset files before any model work begins.


In [ ]:
from pathlib import Path
import json

train_file = Path('robot_waiter_ai/datasets/processed/waiter_sft_train.jsonl')
valid_file = Path('robot_waiter_ai/datasets/processed/waiter_sft_valid.jsonl')
eval_file = Path('robot_waiter_ai/evals/evaluation_cases.yaml')

for path in [train_file, valid_file, eval_file]:
    print(path, 'exists=', path.exists())
    if not path.exists():
        raise FileNotFoundError(f'Missing required file: {path}')

def load_jsonl(path: Path):
    rows = []
    with path.open('r', encoding='utf-8') as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(train_file)
valid_rows = load_jsonl(valid_file)

print('Train count:', len(train_rows))
print('Valid count:', len(valid_rows))

print('\nSample train record:')
print(json.dumps(train_rows[0], ensure_ascii=False, indent=2))
print('\nSample valid record:')
print(json.dumps(valid_rows[0], ensure_ascii=False, indent=2))


## F. Model Configuration
Primary smoke-test base model:
- `Qwen/Qwen3-0.6B`

Fallback note:
- If this first smoke test is clearly too weak, the next candidate is `Qwen/Qwen3-1.7B`.

Do not hard-code any Hugging Face token in this notebook.


In [ ]:
RUN_NAME = 'qwen3_0_6b_smoke_test'
BASE_MODEL = 'Qwen/Qwen3-0.6B'
OUTPUT_DIR = Path('/content/garsonbot_outputs') / RUN_NAME
GENERATED_OUTPUTS_PATH = OUTPUT_DIR / 'generated_outputs_qwen3_0_6b_smoke.jsonl'

# Fallback reference only:
# FALLBACK_MODEL = 'Qwen/Qwen3-1.7B'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output directory:', OUTPUT_DIR)


## G. Tokenization / Formatting
This section loads the tokenizer and converts the repo's existing `messages` records into training text.

TODO:
- If the exact Qwen3 chat template changes or behaves unexpectedly, adjust the formatting logic here before running a real experiment.


In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def render_chat_example(row: dict) -> dict:
    # TODO: adjust this if the model's preferred chat template changes.
    text = tokenizer.apply_chat_template(
        row['messages'],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {
        'text': text,
        'messages': row['messages'],
        'metadata': row.get('metadata', {}),
    }

train_dataset = Dataset.from_list([render_chat_example(row) for row in train_rows])
valid_dataset = Dataset.from_list([render_chat_example(row) for row in valid_rows])

print(train_dataset)
print(valid_dataset)
print('\nRendered preview:')
print(train_dataset[0]['text'][:1200])


## H. LoRA/QLoRA Configuration
These are conservative smoke-test settings and may need adjustment depending on GPU type, VRAM, and tokenizer/model behavior.

This notebook saves adapter/checkpoint artifacts only.


In [ ]:
import torch
from peft import LoraConfig
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

use_qlora = True

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
) if use_qlora else None

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map='auto' if use_qlora else None,
)
model.config.use_cache = False

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'up_proj', 'down_proj', 'gate_proj'],
)

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR / 'trainer_output'),
    num_train_epochs=3,
    learning_rate=2e-4,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    max_seq_length=1024,
    logging_steps=5,
    eval_strategy='steps',
    eval_steps=10,
    save_strategy='steps',
    save_steps=25,
    seed=42,
    report_to='none',
    dataset_text_field='text',
    bf16=torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8,
    fp16=not (torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8),
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

print('Trainer configured. Review settings carefully before running training.')


## I. Training Cell
Review the model, tokenizer, formatting, and hyperparameters before running this cell.

This cell is intentionally included so the notebook can be executable in Colab, but it is not executed in this repo task.


In [ ]:
# Review before running.
train_result = trainer.train()
train_result


In [ ]:
# Save adapter/checkpoint only after training.
adapter_dir = OUTPUT_DIR / 'adapter'
tokenizer_dir = OUTPUT_DIR / 'tokenizer'
metrics_path = OUTPUT_DIR / 'train_metrics.json'

trainer.save_model(str(adapter_dir))
tokenizer.save_pretrained(str(tokenizer_dir))

import json
with metrics_path.open('w', encoding='utf-8') as handle:
    json.dump(train_result.metrics, handle, ensure_ascii=False, indent=2)

print('Saved adapter to:', adapter_dir)
print('Saved tokenizer to:', tokenizer_dir)
print('Saved metrics to:', metrics_path)


## J. Simple Post-Training Generation
This section generates outputs for `evaluation_cases.yaml` and saves them as `generated_outputs_qwen3_0_6b_smoke.jsonl`.

Each JSONL record includes:
- `case_id`
- `response`
- `backend_name`
- `metadata`


In [ ]:
import yaml

with eval_file.open('r', encoding='utf-8') as handle:
    eval_payload = yaml.safe_load(handle)

evaluation_cases = eval_payload['evaluation_cases']
print('Evaluation cases:', len(evaluation_cases))


In [ ]:
def generate_response(user_text: str, max_new_tokens: int = 128) -> str:
    messages = [
        {
            'role': 'system',
            'content': 'Sen nazik, dikkatli, menüye sadık ve Türkçe konuşan bir garson robot asistanısın.',
        },
        {'role': 'user', 'content': user_text},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        generated = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    completion_tokens = generated[0][model_inputs['input_ids'].shape[1]:]
    return tokenizer.decode(completion_tokens, skip_special_tokens=True).strip()

generated_rows = []
for case in evaluation_cases:
    response = generate_response(case['user'])
    generated_rows.append(
        {
            'case_id': case['id'],
            'response': response,
            'backend_name': 'qwen3_0_6b_smoke_colab',
            'metadata': {
                'model': BASE_MODEL,
                'run_name': RUN_NAME,
                'user_prompt': case['user'],
            },
        }
    )

with GENERATED_OUTPUTS_PATH.open('w', encoding='utf-8') as handle:
    for row in generated_rows:
        handle.write(json.dumps(row, ensure_ascii=False) + '\n')

print('Saved generated outputs to:', GENERATED_OUTPUTS_PATH)
print('Preview row:', json.dumps(generated_rows[0], ensure_ascii=False, indent=2))


## K. Download / Export Section
Use this section to package adapter outputs and download the generated benchmark JSONL.

After downloading `generated_outputs_qwen3_0_6b_smoke.jsonl`, copy it back into the repo workspace and score it locally with:

```powershell
.venv\Scripts\python.exe -m robot_waiter_ai.evals.generated_output_adapter --outputs <path>
```


In [ ]:
import shutil
from google.colab import files

adapter_zip_base = str(OUTPUT_DIR / 'adapter_export')
adapter_zip_path = shutil.make_archive(adapter_zip_base, 'zip', root_dir=str(adapter_dir))

print('Created adapter zip:', adapter_zip_path)
print('Generated outputs file:', GENERATED_OUTPUTS_PATH)

files.download(adapter_zip_path)
files.download(str(GENERATED_OUTPUTS_PATH))


## Final Notes
- If `Qwen/Qwen3-0.6B` looks too weak, document why before moving to `Qwen/Qwen3-1.7B`.
- Keep the deterministic baseline as the reference system until a learned model earns trust.
- Do not treat this notebook as proof of production readiness.
